In [1]:
import torch
torch.cuda.is_available()

True

In [ ]:
!pip install timm
!pip install torch torchvision
!pip install scikit-learn
!pip install matplotlib
!pip install tqdm

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import numpy as np

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'computersc-newxii-pdf-syllabus-vsp22.html', 'computersc-newxii-pdf-syllabus-vsp22.html.gdoc', 'Untitled form (2).gform', 'Untitled form (1).gform', 'Untitled form.gform', 'Texus ', 'Manoj Padmanaabhan S Resume 2 (1).pdf', 'Coffee_Project']


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/Coffee_Project"))

['models', 'results', 'dataset', 'dataset_classification']


In [ ]:
print(os.listdir("/content/drive/MyDrive/Coffee_Project/dataset"))

['archive (6).zip', 'BRACOL-ORIGINAL-ANNOTATIONS', 'BRACOL_REVIEWED_ANNOTATIONS']


DATA SET ZIP extraction

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/Coffee_Project/dataset/archive (6).zip"
extract_path = "/content/drive/MyDrive/Coffee_Project/dataset/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Done ✅")

KeyboardInterrupt: 

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/Coffee_Project/dataset"))

['archive (6).zip', 'BRACOL-ORIGINAL-ANNOTATIONS', 'BRACOL_REVIEWED_ANNOTATIONS']


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/Coffee_Project/dataset/BRACOL_REVIEWED_ANNOTATIONS"))

['BRACOL_REVIEWED']


In [ ]:
print(os.listdir("/content/drive/MyDrive/Coffee_Project/dataset/BRACOL_REVIEWED_ANNOTATIONS/BRACOL_REVIEWED"))

['README.dataset.txt', 'README.roboflow.txt', 'data.yaml', 'test', 'train', 'valid']


In [4]:
import yaml

data_yaml_path = "/content/drive/MyDrive/Coffee_Project/dataset/BRACOL_REVIEWED_ANNOTATIONS/BRACOL_REVIEWED/data.yaml"

with open(data_yaml_path, 'r') as f:
    data = yaml.safe_load(f)

print("Classes:", data['names'])

Classes: ['Cercospora', 'Miner', 'Phoma', 'Rust']


YOLO format ➝ to Image Classification format

In [ ]:
import os
import cv2
import shutil
from tqdm import tqdm

# Base path
base_path = "/content/drive/MyDrive/Coffee_Project/dataset/BRACOL_REVIEWED_ANNOTATIONS/BRACOL_REVIEWED"

# Output classification dataset path
output_base = "/content/drive/MyDrive/Coffee_Project/dataset_classification"

# Class names
classes = ['Cercospora', 'Miner', 'Phoma', 'Rust']

# Create classification folder structure
for split in ['train', 'valid', 'test']:
    for cls in classes:
        os.makedirs(os.path.join(output_base, split, cls), exist_ok=True)

def convert_split(split):
    image_folder = os.path.join(base_path, split, "images")
    label_folder = os.path.join(base_path, split, "labels")

    for img_file in tqdm(os.listdir(image_folder)):
        if not img_file.endswith(('.jpg', '.png')):
            continue

        img_path = os.path.join(image_folder, img_file)
        label_path = os.path.join(label_folder, img_file.replace(".jpg", ".txt").replace(".png", ".txt"))

        if not os.path.exists(label_path):
            continue

        image = cv2.imread(img_path)
        h, w, _ = image.shape

        with open(label_path, 'r') as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            parts = line.strip().split()
            class_id = int(parts[0])
            x_center, y_center, bw, bh = map(float, parts[1:])

            x1 = int((x_center - bw/2) * w)
            y1 = int((y_center - bh/2) * h)
            x2 = int((x_center + bw/2) * w)
            y2 = int((y_center + bh/2) * h)

            cropped = image[y1:y2, x1:x2]

            class_name = classes[class_id]
            save_path = os.path.join(output_base, split, class_name, f"{img_file.split('.')[0]}_{i}.jpg")

            cv2.imwrite(save_path, cropped)

print("Conversion Completed ✅🔥")

Conversion Completed ✅🔥


In [ ]:
print(os.listdir("/content/drive/MyDrive/Coffee_Project/dataset_classification/train"))

['Cercospora', 'Miner', 'Phoma', 'Rust']


In [ ]:
!pip install timm

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm

In [5]:
import os

print("Train Cercospora images:",
      len(os.listdir("/content/drive/MyDrive/Coffee_Project/dataset_classification/train/Cercospora")))

Train Cercospora images: 133


In [ ]:
print("Valid Cercospora images:",
      len(os.listdir("/content/drive/MyDrive/Coffee_Project/dataset_classification/valid/Cercospora")))

print("Test Cercospora images:",
      len(os.listdir("/content/drive/MyDrive/Coffee_Project/dataset_classification/test/Cercospora")))

Valid Cercospora images: 39
Test Cercospora images: 29


In [ ]:
print(os.listdir("/content/drive/MyDrive/Coffee_Project"))

['models', 'results', 'dataset', 'dataset_classification']


In [ ]:
import os

base_path = "/content/drive/MyDrive/Coffee_Project/dataset/BRACOL_REVIEWED_ANNOTATIONS/BRACOL_REVIEWED"

print("Train images:", len(os.listdir(base_path + "/train/images")))
print("Train labels:", len(os.listdir(base_path + "/train/labels")))

Train images: 1213
Train labels: 1213


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Image transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Validation dataset
valid_dataset = datasets.ImageFolder(
    root="/content/drive/MyDrive/Coffee_Project/valid",
    transform=transform
)

# Validation loader
valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False
)

print("Validation loader ready ✅")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Coffee_Project/valid'

In [ ]:
import shutil

shutil.rmtree("/content/drive/MyDrive/Coffee_Project/dataset_classification")
print("Old classification folder deleted ✅")

Old classification folder deleted ✅


In [ ]:
import os
import cv2
from tqdm import tqdm

base_path = "/content/drive/MyDrive/Coffee_Project/dataset/BRACOL_REVIEWED_ANNOTATIONS/BRACOL_REVIEWED"
output_base = "/content/drive/MyDrive/Coffee_Project/dataset_classification"

classes = ['Cercospora', 'Miner', 'Phoma', 'Rust']

for split in ['train', 'valid', 'test']:
    for cls in classes:
        os.makedirs(os.path.join(output_base, split, cls), exist_ok=True)

def convert_split(split):
    image_folder = os.path.join(base_path, split, "images")
    label_folder = os.path.join(base_path, split, "labels")

    for img_file in tqdm(os.listdir(image_folder)):
        if not img_file.endswith(('.jpg', '.png')):
            continue

        img_path = os.path.join(image_folder, img_file)
        label_path = os.path.join(label_folder, img_file.replace(".jpg", ".txt").replace(".png", ".txt"))

        if not os.path.exists(label_path):
            continue

        image = cv2.imread(img_path)
        h, w, _ = image.shape

        with open(label_path, 'r') as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            parts = line.strip().split()
            class_id = int(parts[0])
            x_center, y_center, bw, bh = map(float, parts[1:])

            # Convert YOLO to pixel coords
            x1 = max(0, int((x_center - bw/2) * w))
            y1 = max(0, int((y_center - bh/2) * h))
            x2 = min(w, int((x_center + bw/2) * w))
            y2 = min(h, int((y_center + bh/2) * h))

            if x2 <= x1 or y2 <= y1:
                continue  # skip invalid crops

            cropped = image[y1:y2, x1:x2]

            class_name = classes[class_id]
            save_path = os.path.join(output_base, split, class_name, f"{img_file.split('.')[0]}_{i}.jpg")

            cv2.imwrite(save_path, cropped)

print("Conversion Completed ✅🔥")

convert_split("train")
convert_split("valid")
convert_split("test")

Conversion Completed ✅🔥


100%|██████████| 180/180 [00:11<00:00, 15.66it/s]


In [ ]:
print("Train Cercospora:",
      len(os.listdir("/content/drive/MyDrive/Coffee_Project/dataset_classification/train/Cercospora")))

Train Cercospora: 133


*vit*  model traning


In [2]:
import torch
import torch.nn as nn
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [12]:
train_path = "/content/drive/MyDrive/Coffee_Project/dataset_classification/train"
val_path   = "/content/drive/MyDrive/Coffee_Project/dataset_classification/valid"
test_path  = "/content/drive/MyDrive/Coffee_Project/dataset_classification/test"

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
import os

print(os.listdir(base_path + "/train"))

['images', 'labels.cache', 'labels']


In [13]:
train_dataset = datasets.ImageFolder(train_path, transform=transform)
val_dataset   = datasets.ImageFolder(val_path, transform=transform)
test_dataset  = datasets.ImageFolder(test_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

Classes: ['Cercospora', 'Miner', 'Phoma', 'Rust']
Train size: 5796
Validation size: 1580
Test size: 847


In [14]:
model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=len(train_dataset.classes)
)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [17]:
epochs = 15

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch 1, Loss: 0.13017302399160913
Epoch 2, Loss: 0.07356737462172393
Epoch 3, Loss: 0.04310206824024195
Epoch 4, Loss: 0.03879743525780773
Epoch 5, Loss: 0.020391391134610755
Epoch 6, Loss: 0.024082669622747653
Epoch 7, Loss: 0.017587827770862183
Epoch 8, Loss: 0.003920831385969974
Epoch 9, Loss: 0.001617911137285321
Epoch 10, Loss: 0.0023143080244677765
Epoch 11, Loss: 0.0017838125154187535
Epoch 12, Loss: 0.0016310757356475254
Epoch 13, Loss: 0.0013415540835069663
Epoch 14, Loss: 0.001518161242828043
Epoch 15, Loss: 0.0015141323481011946


In [21]:
import os

save_dir = "/content/drive/MyDrive/Coffee_Project/saved_models"

# Create directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

In [22]:
model_path = os.path.join(save_dir, "vit_coffee_92_53.pth")

torch.save(model.state_dict(), model_path)

print("Model saved successfully at:", model_path)

Model saved successfully at: /content/drive/MyDrive/Coffee_Project/saved_models/vit_coffee_92_53.pth


In [18]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

val_acc = accuracy_score(all_labels, all_preds)
print("Validation Accuracy:", val_acc * 100)



Validation Accuracy: 92.53164556962025


Accuracy for vit model = 93

In [6]:
import os
from collections import Counter

train_counts = {}
base_train = "/content/drive/MyDrive/Coffee_Project/dataset_classification/train"

for cls in os.listdir(base_train):
    train_counts[cls] = len(os.listdir(os.path.join(base_train, cls)))

print(train_counts)

{'Cercospora': 133, 'Miner': 171, 'Phoma': 1310, 'Rust': 4182}


In [8]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_path = "/content/drive/MyDrive/Coffee_Project/dataset_classification/train"
val_path = "/content/drive/MyDrive/Coffee_Project/dataset_classification/valid"
test_path = "/content/drive/MyDrive/Coffee_Project/dataset_classification/test"

train_dataset = datasets.ImageFolder(train_path, transform=transform)
val_dataset = datasets.ImageFolder(val_path, transform=transform)
test_dataset = datasets.ImageFolder(test_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 5796
Validation size: 1580


In [ ]:
print(len(train_loader))

182


In [ ]:
import torch
import torch.nn as nn
import timm
import torchvision.models as models

**NEW model train for HADTF**

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [33]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [34]:
import os

print(os.listdir("/content/drive/MyDrive/Coffee_Project"))

['models', 'dataset', 'dataset_classification']


In [ ]:
train_path = "/content/drive/MyDrive/Coffee_Project/dataset_classification/train"
val_path   = "/content/drive/MyDrive/Coffee_Project/dataset_classification/valid"

train_dataset = datasets.ImageFolder(train_path, transform=transform_train)
val_dataset   = datasets.ImageFolder(val_path, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 5796
Validation size: 1580


In [35]:
class HADTF(nn.Module):
    def __init__(self, num_classes=4):
        super(HADTF, self).__init__()

        # CNN Backbone
        self.cnn = models.resnet50(pretrained=True)
        self.cnn.fc = nn.Identity()

        # ViT Backbone
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True)
        self.vit.head = nn.Identity()

        self.cnn_dim = 2048
        self.vit_dim = 768

        self.alpha = nn.Parameter(torch.tensor(0.5))

        # Add dropout for better accuracy
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.cnn_dim + self.vit_dim, num_classes)
        )

    def forward(self, x):
        cnn_features = self.cnn(x)
        vit_features = self.vit(x)

        fused = torch.cat((
            self.alpha * cnn_features,
            (1 - self.alpha) * vit_features
        ), dim=1)

        return self.classifier(fused)

model = HADTF(num_classes=4).to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-5)

In [ ]:
epochs = 15

for epoch in range(epochs):

    # ---- TRAIN ----
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    # ---- VALIDATION ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {train_loss:.4f} "
          f"Val Accuracy: {val_accuracy:.2f}%")

Epoch [1/15] Loss: 0.2511 Val Accuracy: 89.94%
Epoch [2/15] Loss: 0.1555 Val Accuracy: 91.52%
Epoch [3/15] Loss: 0.1207 Val Accuracy: 90.89%
Epoch [4/15] Loss: 0.0980 Val Accuracy: 93.23%
Epoch [5/15] Loss: 0.0748 Val Accuracy: 92.22%
Epoch [6/15] Loss: 0.0594 Val Accuracy: 92.03%
Epoch [7/15] Loss: 0.0485 Val Accuracy: 92.78%
Epoch [8/15] Loss: 0.0258 Val Accuracy: 93.23%
Epoch [9/15] Loss: 0.0210 Val Accuracy: 93.92%
Epoch [10/15] Loss: 0.0328 Val Accuracy: 91.33%
Epoch [11/15] Loss: 0.0243 Val Accuracy: 93.29%
Epoch [12/15] Loss: 0.0202 Val Accuracy: 92.34%
Epoch [13/15] Loss: 0.0160 Val Accuracy: 92.22%
Epoch [14/15] Loss: 0.0116 Val Accuracy: 92.53%
Epoch [15/15] Loss: 0.0244 Val Accuracy: 92.15%


In [ ]:
torch.save(model.state_dict(),
           "/content/drive/MyDrive/Coffee_Project/models/hadtf_improved.pth")

print("Model saved successfully ✅")

Model saved successfully ✅


In [36]:
model = HADTF(num_classes=4)
model.load_state_dict(torch.load(
    "/content/drive/MyDrive/Coffee_Project/models/hadtf_improved.pth",
    map_location=device
))
model = model.to(device)
model.eval()

print("Model loaded successfully ✅")

Model loaded successfully ✅


In [37]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Final Validation Accuracy: {accuracy:.2f}%")

Final Validation Accuracy: 84.49%


In [38]:
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [39]:
test_path = "/content/drive/MyDrive/Coffee_Project/dataset_classification/test"

test_dataset = datasets.ImageFolder(test_path, transform=transform_val)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [40]:
print(train_dataset.classes)

['Cercospora', 'Miner', 'Phoma', 'Rust']


In [41]:
test_accuracy = evaluate(model, test_loader, device)
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 95.28%
